# fractal autoresearch

Can a coding agent discover a better way to fit the Mandelbrot set?

This is a small Colab version of [fractalsearch](https://github.com/MaxRobinsonTheGreat/fractalsearch), which is inspired by Karpathy's [autoresearch](https://github.com/karpathy/autoresearch). It is a separate practice project: nothing here submits to the competition.

## The whole idea

- `harness/` is fixed: target, timer, evaluator, and MSE.
- `solutions/candidate.py` is the one file the coding agent changes.
- `AGENT.md` describes the experiment loop.
- `runs.jsonl` is the lab notebook.

One hypothesis, one edit, one run, keep the evidence, repeat.

## Start a GPU runtime

In Colab choose **Runtime → Change runtime type → T4 GPU**, then run this cell.

In [ ]:
import torch

assert torch.cuda.is_available(), 'No GPU found. Switch the Colab runtime to a GPU.'
gpu = torch.cuda.get_device_name(0)
print('GPU:', gpu)
if 'T4' not in gpu:
    print('This will run, but compare scores only with runs on the same GPU.')

## Build the workspace

The setup code is collapsed because it is plumbing. It writes the original-style `harness/` and `solutions/` project into the Colab VM. No clone needed.

In [ ]:
# This cell contains the project files so the notebook does not need to clone a repo.
import base64, io, os, subprocess, zipfile
from pathlib import Path

PROJECT = Path('/content/fractalsearch')
FIXED = Path('/content/fractalsearch_fixed')

if (PROJECT / '.git').exists():
    print('Keeping the existing research workspace:', PROJECT)
else:
    PROJECT.mkdir(parents=True, exist_ok=True)
    FIXED.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(base64.b64decode('UEsDBAoAAAAAAG5WKF0AAAAAAAAAAAAAAAAIABwAaGFybmVzcy9VVAkAA/AgoGrwIKBqdXgLAAEE9QEAAAQAAAAAUEsDBBQAAAAIANtVKF1Rq2nDVAQAAJ4JAAATABwAaGFybmVzcy9jb2xvcm1hcC5weVVUCQAD3h+gaosgoGp1eAsAAQT1AQAABAAAAACdVttu4zYQfddXTJUXaeu4ul9cpEDaDVpgu2iRdtsHIwhom7bVlUmBpJJ1g/x7h0NZvjR9qV7GJoeHM2dGc+T7/u+N2MOKd1ysuFjur9eKc1jKVqod6zQEQsKOma6Vpm0W4dTzWNe1+2Cj2H4Cgu14COTIILl+D+tWMgNM4S40AubRJH4AI4EJCH6awJ8TSEPoG2EquP/xe2h2bMNhsffaRnCmWnvIcNXJlplGbGDBzTPnAsG3TKzWfQtyjVjLrVQuRj2FT5qv7BXKZqDAbDmsFVsa1np4BrhS6LzlzFCYGJT14E+s7ZnBnU7xp4Y/g/W1OyumtwvJ1Grq+b7vec2uk8qA6HfdHpgG0XneFdy6GLSRnZ5B0EndmEaKMecJBGoCmwkswnEtnMJt1yn5BbO2ztomg1ci3JFh6Lha8s70rD1WAZGkxFSE7DdbWOPFT41Gj+ZvApp6jx9vf/0NbuDFA3z8Rqy5EtKfwZwW7BNE0yiaOBNPwJqITJyFIa3H+WBKWs9Sa9Kkxu1TlNShpHVBblViTZbGA0rmUIoks+txkRNKVV2gFA6lSimWxKEkeT2glA6lLmg7q6PhrguU2qHUFe2XLvI4TwklPiRcV9UE4kPCRZYeUB6c8bELd+x/spU5GnJKNy3i/2ArJx6iOnP5ZOds5UV9wlYeXeY5sFXaRJAmV5osSy/YcrFkEXml2WXlDmy5yhUFsZXl8SVbtF7XlHd5RDmwZXq1eKO33Ol4KFXpCpomB7oSF2NSUopZQm5VHV3EmOTRaRcWDq6OigEnHXIdqKocXJFf5poP8WQlkVFn9bGbXX86nMp1XuUKE1FhTnHK/KSH7EWujkU54FTlGU4au/TT8z6IR36GeKLYtU2cXLK7abR5tOPKMgxwBYuWLT/D9Xc44FbWSMUEDk389bxtDIfgZHiMh8O3anPo4dEMORw4Tx2ZZ/tvcUqcvIFDHRWML1qapm/jOE5Hv/wc58DVYE/NCVevdgq/52vWt0ZDbzWALZXUmoY4Ttm/+NJMvT9uf/509/gDjkecjuNcpECuaJAapjbcwDdWB1bN0g1xq0rau7u//+V+PHusy/Es3sQWWra94ddnOgOBltCYYUyjrHWoE6HneSuO+jUK6AzVZCpWJJdOTGcoKQqvO8Yd2kof3WauS3z/I97CBoE9VV0ntlZcncw6iUWl0VvWYbe8sxdP6Y8V46kVOQvZrCkAENJYxSJFmY1VU6zRHD7w/Z1NM1j7vfgs5LMYNQpe7Omv1Ou3KNVPHF5a5CsglPDVd+24wcQwk2XbdAFapini4VNiZfYdv8FlSqbIQuoJKrw7TVqLCIQ5t7c90Dpqr8N1aHM9jx6oPCT2dOrBIWCsl67xv11tdYMPlhs6pDYLd0YbfA+DOf5y3ygBqjvePSHY+QwtnrVgSwtGL2mQhvglwL40+uY6HuA4jk/8GLKw7yDJ82kEX9uXIEQ+LAWWGapc6P0DUEsDBBQAAAAIANtVKF1Wc5kHKAsAAOIZAAAWABwAaGFybmVzcy9ncm91bmR0cnV0aC5weVVUCQAD3h+gansgoGp1eAsAAQT1AQAABAAAAAClWNty48YRfedXdEmVLCATEElbTlkbqSLv0raqrNWWVr5sVAprCAyJ8YIYGBfxEmfLH5EfydO+7w/kH/IlOT0zAMFd0a5KWCWRxMz0dJ/uPt3Ng4ODrwtdZzFVRV0lNNMFzQoRVSItpSiihP7z6z+pSiSVKpuneNN1EUnSs84BXq5EMZdV2OuNH2Sxxra0rpTOKIWUrDRbFiLPIYTIK6RI+6QWYu5TcO7O0oNIa0kUy5nKZEyJLCTkXV5dfXd78eW349NWaAlRaxzPdVGRgO6QBwMShQUd16ns07SuaFGXFb0Yfz++4cdqhiNVv8f7WRvJ1wmj40JWhYrIK6WkBOrKsjx2yzLM1z5BcKQXeV1BLTEXKoNgVYX0LBHZHCZB6Bo78FhkFe0gYs1R5h6VQaiKIdXKE4WYqlRVaxJRocuSRJpSqudzXFPUWcnWz2ita/JYYFIvRObTku+oNEV8uaRlIipWcCoZ25mqyJPhPKRCLKGjLKyJEXwMhCGyXGgNv2W6WECZDa7CU5qqTMBtC7mYyqJMVO73LaDK+u76xZjyVMDzuDnWxvrn/NaLZaqmfI1M18YZAIHdgjCSZcJ2AIk5PCIjUZeScgHwsFSnFSsNHCSAaeBI2eW3HyHnRXVRyKzyTymXhYI3IwYqiBVDHuEQ4RWQLCORy6BSLLQ1fqlgMFuhi6liuYV6wMqDpM0TOqN4cxxHCGcRvWH/pjqblyrGYt9IJVormcaMrqDmwqDSQQkdZYm7YDvCFpJ+2fzy/h30wjsd49uTX4jgHRNuRjOAI/NGWc4HXAhABU05B2XcJ2xLZQDsEMQvxzeX188vn7nMUBndDfo0vD91evErTwRAxd1fjm8v6P07hsWL/c4OF4dnNAhP6BP+//4dktkb/fvX9+/Mcd/s86yKMTSXq4pRLh4VMwwHtPflZRL5T1tRKmOcfGfyd1mq3kjYu9CZrnQGN9p4DNhbzR0m8KygUlR1YRKGQ0uucl3iMxOEU60EDUXYIxloabmHIcR5YfPQQMvBbZJoynIA78+1zKI1s4/KZiRK+A+fB37o5N66tNqG9ykiimAWX8ZEU1gWhExHmE+sYia1U7kyeY3LVOlEqqxCCCMgkeZrJhoEYsufhicZAs053ITIlaiiRNoUPHjZRL71chv8Bxy7WQlJC9zhLotFmUw17jjGpkpFx+AO2DItdDVJxTRMqkVKngkajozBCSLrhr0rPwNGKtV1BSwODg56PUe0CPOk+VxplIZe75CCIKDvlVwix7JYL/n7//GCwL9qEEJMuB5RkgjF5UgvLKGqecJRcheMwpM+wvB+dRcMwyE+Du/ZCQvk6KktVrKCN2rkVwGZ7IVULxldlZdcLVDX3r7/F86NzCKWpCPQKFUmK13aVjo/nuqqggoynksw5a1lRRZLEfyJozG9/eIPtIHmgdF7yv8t3yiUB5G6jeQFg/BPJ32EGRJdmy3LRKdGXzDyEkJLBVpEgWECcrEFLjMMJpArXCEWyBKVheQ15ccKsg6wVaHcLQuQy4UBAb7DvgJWIw1pS75PW9IXncjvWy4PypSZzt4Q+r0fry5f9OnHq4sfETfwyeewDJCe9F6bhdduYRiO+PmoCRfH7nwjCi+qzf8cND3cMHk+fnn7De4ZDRwtHe6pAnhGXoIYgiPOzqlEAiI8tvQQywpx7/e6OdHy2uHjdWfLJKeOiR0RH1kedtv83vjVs4uX48mN5U/kWFdT1OpY1aVptVJGp8lA1lNEkWHAtvS0Naf36ur6+vabF+NXryD2ZGB5+ZC8VM5FtH5KdYaCG9N0bQKkNcCybJ/eyLwy/GMp2PP9Xm/y7fXXI0jjZA/ZhFE48Hu3P1xPXl4yyLjjyC7mqtdD8Z81pxno8tQyQ3grsxLlwzR3nQe2cIFUvjUa7qkC5c81yKujXyFnyDI2nUsT27FAPhmvoltgkUNw5RCrnlUDn7bY+Ch6Qz+kq/YyFFKz7Snrx+UM9G/P2XJg2fcFMqOLn2tB9gRC0/86A817IVErMnNDYP5DwY/1Y/UY5F7vLxarTE/miAjPN/BueduLtC7iDzDuY8NqEgOq5JQrDJzUZsV+/J/ZbnY3Lh6xx8CP7oTLEFe7XOOK0oF+aVILXLOhPwe0+dsIlkQgHgVCM91UQ02PNlxPTcG28W9IZ451WyrNmX2NluuzjnbarD5fCxUhZqbTmLuqBGPNPOlteySbmUe2PzK0L3io4b5qfwvWgmNasJBeSQOaEWtnDXTDEZoQ03y7Wj6rwdLb7tXBdVHMy23n1vjSA1OOfKqMfxji7nAU0kXGxPSgInkcV+scBN0OIojiWapF9flnYSu1Ewt7SdCpc2OCs6MRNPGtxE9HHX26w1m57UFDujSdnQsJl0m78W8VB/DW2NB+d6nRPr47RUG8Dyvt2VB1RtmWUS129g337YuwzT523RdgZBDt6qZd3UgMWhNuQr3IrsV7Fnfa2sPfHB28zZPJgEuGb6HdTAD+70k9hFYchJDjmsr4g1mh9ZrTc79UswHT3INs1zWmWC8KUeVyeTe475MJnzO7ONUaIWbdcWbfnOocwjyooiShyHttPPnbODmk51sUCmlpEW6uS0MFX46/ur4ZGzs26MBi0zkDoL9nnwz/wSXkaDPJjvAgM8TXiu34Ycmjv2fs6buSs8EfdhiqhOab7WSy59jGHAIh4eN2czOXnDm0/gjfhWJaej6dU1Ojt9vVrDkRYsL3/O7g5dx85zbcQ+am/bKzL/5oY/zIzsZ9jWJv3Y6uMtysmfXH1JmCON5YL8JF++izJd3Tx5nUOmQD9mGDWHGLj4tB+zzuLIRRKhb5BL23N5TBaOB24n4WwPuPnH+YeflB90Q4GAyGvo/LjWynfTPpdLmYEGQ6Nh3+I3ze/4D3G24HLYGvQBsfFYIdtVjdXUM+HfjWko/H5/YkT9GuNTqysrtH7oyn7vdNzIe0Oyqb1qMZXrrtg+ssdnnv05Hv2q+FeCPRL6gYjFeuTA/Q5w5/7T66LD+I8vqg39vVYQVjz2wbv0Kqn3Ev36c1P7U9/JqfciO/v5WwdQRURkaBI776N0oaaTaazXzYjo1NSdLLYCF+4kO8SRiJa2PNyjfHeQhHkV4B1ALDzpFZ4ixnPuMnmNeCczuw2bqgwU5lrrO45NGaZRijgnO2Eu0YC+Wf7h7saI5AgixM47MqOC944ORxMTVIBecMUuiqp2FW20CBulVkf5jEHTtK4zvYkQd9AGQu2y2Rq7KlrxRTXy4i6fFd1iFWykdMzQfXjxxcmyNrc9wqsXuQgw4AGaDODAxG1Bwb56tW3AKmmXhal9ACfwZxJN7ZgfrpwN8JTZsGFXo9726+CguLihcMgeN83f3OFUgtzoYctod0W9SgojKXEfDlGtf8lNIJCvKWKsbYe0yJNI54i0Q6+QJN0Y1FlJUsMcfpOo0hk39cNI0kJutcrWRqB1yeJgrz45lpIzCOgwUNXtT8FOx+v0H3mD2pDHXpAh1W2Lt49XL87BbIeGbKDYhzhcnKe22/c5Y0iWjNsaloNX4kA/02by7M9sAEKKOwtSmkH4zh/KtzokuZsUVb5QOr/NZEI9GaGTZtHblh4c5LjpZwxgj4Gzj7Dk3/KTnvmPi1kWuX3E4/bILU+oGHwpU35DxDo+z2IgUtSP5OYGxJaefSDwPyA52agVJwBzepM8U/aXmZg3EuM+6HQD0u7r5uHkC1F+h3PkD69xjLywxNuWu44Yn1omlnH2WpBo+6zRU+ZOR01DtrPz2auJxoHEZgrdq2vsBwJ75sfmPb6862odm2E3b7EpE5YJtu/wVQSwMEFAAAAAgA21UoXRoQRhz8BgAAqhEAABQAHABoYXJuZXNzL2ludGVyZmFjZS5weVVUCQAD3h+gansgoGp1eAsAAQT1AQAABAAAAACtV82O2zgSvvspCM1FysjqdA578MDBJEF20MB0Z5F05hI0ZFqibWEoUkNS3W0EAfYh9gnnSeYrkvqx08FiNuuLLbqqWPXVVz9KkuT2IFillTO8cmwr3IMQijkcGmEFN9WBHbhRwlrGVc04q/DV1NwJZrXsXaNVsVi8Ys+Gp2essUwrwXjXGc2h7zTbNc41au/tXsOAkFujHXPc7IVjqRWC7Y3uVe1M7w5Fd8yKxZWju+ASLj66Q1DnLmf75h4ucgaXG+WPm1awbV/DVs4knFaWLm15x1IjuMxZ0/J9xpYvF/HGey57sYINJXrDJb6gidiktqJe7rRpg/uPUKSg6E/byUaJC6n1733HbHUQrciZcBXiv7q+/nj76vWvb1es1kwhNFE3iO8ALHaNFKwJmFa6N1YwvRvRLdiNeBixEpbtdRRebAZI7cUG8vvGOnPEjYRp03YS9yuP6uZDFNz4VAxqrNV1H+4+MdX2Fv49doiVvbt5C29WC4bPEkEGlaUU90LC7rtfP95evbvZwIZ1XFUIWJtReLPtG1mnBCwbXWA7EEmbY7H4ICaO2Istt4IALFvZIcEMIAdIuNKqqZAEJxAUiFUskiRZLHZGt6wsd73rjShLClkbBzIAXp8Uu1jEMyJAkOfbahB89foN0ra1ntmtcAddTwoa0McrBnrHv2Y8ZBzpcIvFopIcAv9s3BsUinh0AS94+fZemMhNPuGuhKgtezhQ4geSghjiviEAT9hKNYVi4kgcmaRiHPxBUcL3vnI28IhEO/gBjhC19JT2AuW1KcYUWFxVNbXwFgFxY5h+UMzylhi8Z4SIE/sjS3vVENdztqWYuTkuNQLygqKGczXvHIotB2Wka5bgbO6NVr0xTdXLHqpFUWTsz3//xxcnqw4aQVIL6DjABNEpx1zutWncoUU1kwcD+UVdDEiG+GuxQ8oBmCtLdAW5yyNsq5CyYg5iGUAs7YrtpKbOgD5Sr8BVx9bseRayRB+yFFXxT/hx+ueJPch4g+nJaXaqUaIgDF1EQkWrwUricXouVgteE+8hOdf78Yl7TzX3QgkkCmWyjsH/MpykIYR1+MqKlquey5LCp15aZwHNH9hyuQx8W37vZ0yPkLyz/h6585XvoZqwNgL1qr5CBQ1jFv6UbQ+AFDv3TYMgxweBYkBJGdHGlt+j80nPrWehpT+bxkGAkzhoqFeCZMSvM/9a/pg+L57n51lafuV5NjmLptmYeexbreWJp7emR3tHo/S+xWkzOSQeDxzdFy7NqtUedC9rlKXuxiCecvkrSF+uz7yfp526yv8x7aEpxJpUvsiyk8jfByfTSmtT208qf3GXRwDwdJdRL4j9hhn0MnTeTsOKxUhRsSZPog6GQP69K8LtZdRPMY/H6lifFsvQMNazmp9KMvoTjLbjIhKdzs4RD8djGBMRwpAo/ZSImATZoU/dCmW18RyZH5wg9ka3Xe/E6BRNRG62DThgjkP4aY8pTf9U4A0wm7q4X0Bs9hRTngqucDqdYzKjtdT7GERrMagwH7zjN9jiJoc7g2Slu+TTjqbP+rO3NTSDbPWP4nL3xd6xzzDxJclRxL09rKkcsnGADpRPMZqzcYa+xmLAgoCgcTotjuOSY+N4fC/+6KkAV2yjeCs2OdvUwlam6fzqgUeahPjqINRUGIpe7V9G32MeYgvi92JzsUF/qbHS7NhR98z223B5DcuVk8ef2OaWUjZtNJMfVNXt2cwiTzxoYFXSK3qsk4jt6NsoEJV+PltMhlTA/4FP7nE1WzmeyIhvN2gX7KGhFSR0nGm5KNhHAtY9xupJswt6OKFu5qlFrWK0SmsGyQ29LmhNbTqj1QMDvANY/sp4W/LfQos5+Z/L5ZrW+ZucvciG2oA0DrLBsqjZfL23BbtSO2EEdWOt5HE0lrrZmvVgQGFsoVJav3j765Uu94bXWcF+86aGHi0bv05/wty4vCtO1hbiVoyt4+7wrTJCwpCVG+2uBlKJ+q0x2Kpn5cjr77MU6+2ExunwY6o8UAtvUo0HiNZzTwaliuvwEkBH9bTBxxr8EOuFQAHUG98H8Nog5IaleF8b9en9q6NOhRy9yJYvfapwAb4vs4x45u2NxRWKt5iKN48Fi+9QsdxQ2lSN5ZQdkNizQvRODIya3Fh73CK6X++S43YVHtOk6mueUHcIx/RYNLbk97yRfEt1hB0IYCVV1yexi/58Qpu4A3436SdoCwFKz1bLcTY+3dtHOd27Ye/0dr416CBXoKYPvBPp8jIrkOG2C/vRZfE8+9ssD/F44c8JNj4nSoIhWc1jms7TLB9V55+EmumgRL+/ITZrtIP07OhL8DX72xVW/d65kR5eh6RzInYpdeVfQp/eM2ZRkl45C5WMfppjcvek2nlG/wJQSwMECgAAAAAA21UoXQAAAAAAAAAAAAAAABMAHABoYXJuZXNzL19faW5pdF9fLnB5VVQJAAPeH6Bq3h+ganV4CwABBPUBAAAEAAAAAFBLAwQUAAAACABWVihdTnFJZVcQAABmKQAAEwAcAGhhcm5lc3MvZXZhbHVhdGUucHlVVAkAA8QgoGrOIKBqdXgLAAEE9QEAAAQAAAAApVn9ctvGEf+fT3GDjBvAJSFKlj0xHWaqWrKtRpZVfTTJuB7kCBxJVPgqDhDFuM70IfqEfZL+du8AApTkOhOORiQOe3t7e/vx2z3HcY5uZFLLSok8UyKUWRRH9KTzpK7iPBMYEUm+ENVSiVLpOqn8weBKy4WaDAQ+9Y0o60wU62oJ6lEqlrLMlNa+ahg3rPTOPK/LWJV+sf4dU8VolOV45+RlpMrps/2heCYSuValdgaDH5ayEnElolxp4c7jWxUNRZymdSVniRJFmVd5mCeeEX7XFye5jHhzF82O52We8sgivlGZSPOoxsx5nCifJ+354rKUcUbL3MSynejP48r1xCqulkIKXhrrybCKQyVmdbRQlXAjNZfQoXg21p5hh88B7TwSF8evD07O34qZDK91lRfiOk4SDV66qsNr1pWcV6oUu9+MtZn8xBcXYV4qyDIRby+OhFxAMl0Z+cu8xulVZQ2J8htMlOLV8Y9HhyJSmVbCqpn2vCjjqBXHPclXICZ2sRYzVWFN7HmJB/xdvjkSqarKOPQ9nrEPGeSN4iUrUgz2LcsqnmPr4o9YsyjVTaxWMJ8MJyZq/o/d6J1v8T+Io+92zNpPfXFQFKDiPZd1WNUlmJUKW8Q+cp7k/0PnWcJ2WZRxVoF2WacyE7pOU1muYZ6XkMSIyGQkGO2VdykSnKpYKuiM3sWZVRNriczrv//+D81YYw+qo8aBIRCHuYD5kVXE8zW9T0UaRyNI9gJGF2cLGK1I5TXMD26zYPHJa6C7LMzTQpZkiP7AcWCtbGpBMK9po0EAQy3yEjvKsAQfjB4MmrFygblaNc/mK4lnPowvaUZJN83vVFbL5neum1962aXX8SKTm6d6BgcJ4YLtyLr9WcVpuziOOVRkpq14VV6GS7sh68bNbjr6FVKLRdWj8nGEqoStqIb+VVy9zDF4Ww1b3xoMvoLfj/COvKpjuI1DixDKqmQGPY++6DO4PD84Pg3+fHX4+ugyuBBTuKRoPl9BTSSKyvJ6sbSWJ7SCD8GM1G2hSmgj40PlwPkyT+QMBFqTsG8Ozg+D749PTpgvvFWIDms547CmNn4+z8vWyzfMseX970dhIqHKex12CHUWKqxGcA/4SEWWF+ZwlThD8BzpgpRaQGMURGDMF3+9Ojg/8sGXHIRdchVnUb4it37iPxW3iG17wjVcxa+7/tPnQ3H67lLsTna9IbGHlP+siRe70iqvk0hEMTZRkrjkLXOKeDLxxQ9xhAMHZ5mFUKZiByaK/e/F1ZtDLI33L+CJ8WJZER2iAlwz2sRfIxz4GoF8MfpOPPlmf0xy7u/ui1+n4rn/5K0o4HYVAuLR3w5OgvOjix9J6eO9/XbgJwywEbobkh0Yon9wcXb08tIbXBwhKGLS3pP9weD8HXY8hcv4BTzIj+Iyk6lyH3rGcdK3CzdGiggCz/MG51enF8Hh8XmHzT8gpEush8KhMOZYqpN3rz9DZYIdaAcDYwjBG2SKS9hHXlfu0W2oCjIHm84KUIASOUYEMpFlGiwR4xJVuuTndTqk00mVpUaohk31GHp2doKkGDTJ1yXJJhSQPTqBxiUNE4SxYxugOtn6MQVSUgfH2FIhvFG61JskG7O3hsqnOGhkh7VsFNFolf6ZNEM2AIJ+3PNpNCCLMdpP8pD9w3VaKNPuwxmKDTeb1e/wM+OGI/F26d9GAJ80g2yobrGsoXXNl6GJ54hrWlZVaYdxjhfvTq4uj9+dOlbvzAsRa2qF8BsCfquS+3jM6jiJHmTAb13PztdqQ2bO+AC84hlCzlFZ5qXbvqXP3PlIWvkk0hqwAYeP/I2zNJxHCUJeIn5uJPxZcKT62S74s6AcnyPnOu32KTnGujleF3Juori3LdjlurAyNWJMRLPWjl3ECGZtSLbJpUVddmlLgPWsDVOwDCqkTVVpwK4boLAh5eLbaRsE+HndPv/ktTb9msPFyCQtirpI57BaikqbUGyDMGV16JYCl9JLQUliTfkCQIS4HSOVZSSpTJI1R9NQhksVTZibnUzxD4OyoFVen12JDOmvCZhPR2mcERGjK0AM5msQ5ZA9TBpUY5iZBAH7BxMCfYscvith7KulMlswWjGHHVt3ZJ5bUMjt5AiE/zTP4ZFIBG8PfgwOj84u3wyF7/seJJfZQunWlzkJadgnQixBoYAU5ZLujcaHwpzH1HyZA7RH1cwCSExmSO6uYdY7ZDM0bKbgwP/E+MPPciwlyRXIAoA7ozisgpmsSOFki5PWboaWy8RAF/8Sx5qXQ8HE073xOBiPxxzwuu+NgSBUkpzvP/ATZe+YgEBJWnDHDWdfL2Wh3o8/WKYd6ycGvmSoS1L5VlK71/fxJAZw5kkfPJ+sCozc0a7nV7ml8a3qenoxkiICurRAE8w1FQguLXFns1aB/WHeNIlj5FVlib3ydDFqVc4xVFMANVNTJTOXSB/TBM+PK5XaiJTKu2QU312vR/aVeAnzP7s4PSe7pwIKtqmM8wGTCMCiOcESVFlcmWlafzomYeNsPoR1xyHjDYpAmBVHlu1fLt6dGi/JUGkSQpczVEPiVwC+6M80Q82JM3jCQZtlQLySa1ujFTorGVSM/TEFOdr5t3hWo909jrioATKXXw9BBaLHDL+RLha7Y3cXAzs0yWu2OiM/Q7EyWjH8IUxLYXAiVFog+Me/mNKDysKhWIJkFKlKxglOekH+6loXzpQshw3PuuK9SwQFmqxV5fniXFF+wwLAEesX/OLs/PjtwflPTYkUsybNRlfYpes2rvidwFaeeuIPoh36FkPPnz/H0c2RCit7djNjCq45ZBRi7soYgjUH7H7Fw54PGJMWAaurf/7Whj864OVMSCTkPdgO/Zb0mw4BD/RFGdFqMDDkJMEna/CLuAoQClMux2EeQC7GlKty3clANmG0NY+PIBFeB3AdBNF+knzvgKdDmEzdjLgMo4fRiKsE+vnm6ODQgaOHq2jK8K03XVfADOW0s9Th0d9Or05OPLhxmEcKeoGQcdHkcMZ1ooV3d4R2spwEahwc1XfQVNz6N3j6kNJUAEDL6K4vc7vglyXMc1Pbk3nZYNY0j+DpIzZucXb6GtWA6U28wu5NEgXHNiSXqtbkoKAwB8ZiYlJSKhmtO6kW7CiuETEV7FnOXQJ+a1Ii1z0jdUtNCG7JVG1TwCZmKoKqVW57Rya3S72c5dSLMaW6MoXTqowriELVK8GaSCao+YZ2CUSaBbU8aHcGJ5qsfXXMOVhjccSlmapWyqTflEIV96KUVuUNbeGXPE93Cpn54nW3Z0MNl4jAQ4gavUk1Lb7ggizm0olQBVlBROGLehFNF6OzI9N90bZrxex2ZBHvdMtzV6M8sOUW0hkOpVmE2kdQFFBKghCOOCk5RFLxV2BcUMjyWgDQczSu586OT5r6/jiVC9W+tYOoTgA4oOCs6E/caiag1M/LFIkCpGHakgKJ/EBFHoMLstn2DVljk7/aVErkCEVFDcfjla3jGdlrrkSsx3zRHJMjs4LTGq84Mny2Seg/MoG85SRI34BTSCPfbAhZPT7tXZalXLthSkghWbvMECEm9eF+V0fBy7cHZwihdO5ur360To2w1MNz2cLxvmAZEv+3rbJx+C9eBLvnNY7Oz9+df8kaHEG67LdjJBlEp/Dh1iCqCrftRNruo76OgbuAvj+qTx5C9zyp9XJ6WdaqwUspQLZrIxsMbdq23vyDclFTZ+aMnkp7/hKnHkWIv+ad63TqzaVKiqnDpS1jmnsqZOcBLqbL7RBU5o7x1Gn5oTQoJcZ1iJRhWta5Cb2AGw/zMwUDuFSouqacvjfc+92wB3mgBrhWo0pp4iP5zKcOtX9UACuDuPflECv2P+s4vObAim3vsFMJTrkvTLeesAvdNEiDWCOKcc1uygUBbgjEB0EiaWpW0DvbWZ8CmOw9JXBGL30WNCBBDT7jQUNqzIeqQwoUmPj82YPT2szXm7Ru+0kbNr1+kgmCBvHKrAaNVihAqM1kNmTQewuLzaPrhHUkHRLGInk8+rEO5A2gH7WNgWdYLAdhqKl7uYNOnOJUEYqY0w/XefTTo/RRNHr05tHbRxcdWrjTnX6T7VYNLTNDDBKq3UCvN36obmNdBfl14zAm31Erot8yMspsyn6L77iLPxUfWxtxzHqAb+bHxnoc2gRybVrgJe+Md+V1KFo3m7S7KWFn1DTabiL1pRkKQmhdVtTPAxvYBndduGdhBvFdZ/QrcroTOr6HeW5/YvclHh0PtZzriH8Jh25DSBS+v2pMjh/MqTpegwNB3lvQwFmSsYNtewKRATmTruFAqsCMB9ywRDWLNc2ITzFATKfCmtzGqroaxq5qDaZ0TtR36L7kXgTMOsyziGio9Nm8NaD8lGGSBfH2waJ4+7SF4nmUuXwa2KrmIpMFYLZBje2tpM7rkm4MMtsiIcgDA/UNqLOXUH7B4CjuNmAs1w1gskCJQwQWSEE00iG1S7ptftNkmdftSwBxvgNcchWvLVsqu4HYwDCm65+8YPi0Wsrqay1WwC/LLupzias5zArPHglLm3kMV4kL9bgp7DK7e1MrarPJMEcCCWVBN0fIfIRzgcpoKepDDAkmZhxJlubuEjC3UqXlaBs2LwjZpUBviURkNggLKolUoYdcTuL1TMmKYzNXzpRoSoUqJqpDrqTbO2GOeF3YZy6afJJz7/955L1ZQ9wPBZrRmdSK7brP6zfCg22LYrV+FiPQ9LCivLG5qHJ7/awhR6zAZJtAT5tOHeWA6SYHmLs333y59sneAQ+3Gvi9GfzGfTJ+MGvR3jp3UFYl1bjJEmmO88yzOLQQpn9uecI32djjBsiZ4P1+y+s/3MOQkO94w6wr8dhqjzXYb+ERNL6vX9vm183Ptdc5RcVtxnsafU0TbkP8UOyb9IyvEz/1OguXJXb1i3K3NeHXBd0tuJ3eWrsX767abCQlfTn5tTPoCsUtm/4p9kW6r7xvF2tL+Id01fWG7l3P5LNCVobI+V0G8Lk2xt0lwxKl6u9bsDuPCwZD31xW+3MqHasAcrneVijYmtb3eSKbx5w/Jp837f93njM8BVtXXd07P3rPd35OtyBd9w2C2xk50o7bsvMovs37ZGZzMPm81gFRYk1izXdY7tx7z1n6w6ZCNcflvkK2OM2rV5Sa+XJmKL5Xa/urvbEZGl7UXD3kHhaPen0Rtpc3PUMnzubOXSeJDRmf2V0DMZ5j7xPNSxZffNtfpetadzhvOdZvyFD3nlZbzjnN0RXrbvn7wGGBeOU8dGKs1ahOC7fphPa2O/wMaL7n00XJrU6boQ+fYOZDZAvqMU33OtF5I/QD5bjN+sZSH9zPZi9m7a3l7q7W3IiDo7yX49ynXpxyW87asiaQ7fyd3MY07NmpUbCOrKE1Cd/obNLw+2iV8rUZ//rDpy36RlmTLfpm/J4ZbLJ3VzDjd+l7AW6yoe+Nf/1h4u/OP22uWR9ykLsQB2bTCtOVB+PE9ZuWa2+SfGCS5EnP7p1E0D6IZpPtSTROs/bundUtASadWd3xu4J+SeJsVsjUiv1/+wxtWNgciV5rHxVu5Y4/p2KDsXapXQSqgIurIOC3QUDNoyCwx2A6SYP/AVBLAQIeAwoAAAAAAG5WKF0AAAAAAAAAAAAAAAAIABgAAAAAAAAAEADtQQAAAABoYXJuZXNzL1VUBQAD8CCganV4CwABBPUBAAAEAAAAAFBLAQIeAxQAAAAIANtVKF1Rq2nDVAQAAJ4JAAATABgAAAAAAAEAAACkgUIAAABoYXJuZXNzL2NvbG9ybWFwLnB5VVQFAAPeH6BqdXgLAAEE9QEAAAQAAAAAUEsBAh4DFAAAAAgA21UoXVZzmQcoCwAA4hkAABYAGAAAAAAAAQAAAKSB4wQAAGhhcm5lc3MvZ3JvdW5kdHJ1dGgucHlVVAUAA94foGp1eAsAAQT1AQAABAAAAABQSwECHgMUAAAACADbVShdGhBGHPwGAACqEQAAFAAYAAAAAAABAAAApIFbEAAAaGFybmVzcy9pbnRlcmZhY2UucHlVVAUAA94foGp1eAsAAQT1AQAABAAAAABQSwECHgMKAAAAAADbVShdAAAAAAAAAAAAAAAAEwAYAAAAAAAAAAAApIGlFwAAaGFybmVzcy9fX2luaXRfXy5weVVUBQAD3h+ganV4CwABBPUBAAAEAAAAAFBLAQIeAxQAAAAIAFZWKF1OcUllVxAAAGYpAAATABgAAAAAAAEAAACkgfIXAABoYXJuZXNzL2V2YWx1YXRlLnB5VVQFAAPEIKBqdXgLAAEE9QEAAAQAAAAAUEsFBgAAAAAGAAYADwIAAJYoAAAAAA=='))) as zf:
        zf.extractall(FIXED)
    with zipfile.ZipFile(io.BytesIO(base64.b64decode('UEsDBBQAAAAIACJWKF3DAJWVLQMAAJYGAAAIABwAQUdFTlQubWRVVAkAA2AgoGpxIKBqdXgLAAEE9QEAAAQAAAAArVVNb9xGDL3rVxA24JNW+U7bReBLkwYBkqBoeylQJJrVULsTj2aE4chrBT30R/QX9pf0cSTXTmtfihxs7Njk4+PjI/eU+mS6bLywSd2B/vrjT0q8PnyMY1X9GqdEn+KOnFCO5IYxxUsmQ1Nwl5zEeOqn0GUXA5kR/7xyg8kxUY+ffGDq3RVbemeCZb9LMVM2ac+5qarTU3qd4hQspcmzVNWGfkzITjMNnJPrtnRpvLOmgL/7+VVDb+ORk1LZcc6cGqS8AouZpIuJFSgQwKVUFjMwPX+4Ee4iiuRkXHBhT7vJFgLItS5TDH6mVqKftI486EBVi3Izzi3hQe21Jh9DzCzNYFvNfs8oTUO0rgfAwaTAIg/auhRfOlg+Lx3Xa+GahNnWBH0Y/U1Le/vkrIK+jIQiBDS76aLl28pB2MO/0sboAtrFr7j0vLZxC8pYS5ZHBkzoHIsiTMIlPHA+xnSh0b/guQIjQMwlIg3wptQBNphRDgDrS3lVXKVWcbo4uhIa+EhHFwI0AZlbgu5Ysmq5jPylEnWY9qOGfmJj75W+vkv3eh3IFKT5JBgdUB8DCGS0n50R9i4wuf6LILUMD2OetxURteOcDxBvM9A6tWbt/EbAL8nQufbb+Linx+dnj9rqSUM/xDTAPEydB0k6zGMEBUElpaiqw1yqxF3uedrQ9wcT9ssYZDAeG5B1Lv3kaTQJvuzvlaapnmk+dxdIN1kLCUyW5f+2t9nIEC94A365rZ4DPA6D0gc5Z9kUH4cy87LTk/c3ZlmK7t1itXsq/EcEOjsjzemWSiB7wlcj4wBwyFt6oWXPT9qvNK9vVrMpe1CZfIZXsU3tPvFIJx8G4e1vv3+AinkS/QQ3f1Tjbk+ukaD6tw296VXtLhnBrGtgqYOzcZ42gZ49/Ce41sO3eHLaL65NOAnzNYKPovnYnvmuXYGTdbjx/sO0Y+zi9RZf5TKmpvpu2YUet7ks5HoXb5SVdZSS41hYyTTg5LrPTEe1kp4DvU7l0aMvfWhcaUU9qoe4qlb3qv8H3NVSnpBiKKNOU85JOU66eqbsSLhFo3y71Mt9gg0M/pL4mByaq/4GUEsDBAoAAAAAACNWKF0AAAAAAAAAAAAAAAAKABwAcnVucy5qc29ubFVUCQADYSCgamEgoGp1eAsAAQT1AQAABAAAAABQSwMECgAAAAAA21UoXQAAAAAAAAAAAAAAAAoAHABzb2x1dGlvbnMvVVQJAAPeH6Bq1SCganV4CwABBPUBAAAEAAAAAFBLAwQUAAAACADbVShdFw5F4UoDAACSBgAAGQAcAHNvbHV0aW9ucy9iYXNlbGluZV9tbHAucHlVVAkAA94foGpxIKBqdXgLAAEE9QEAAAQAAAAAbVVdb9s4EHzXr1jocDgylXiOi/ghgA9oi15RIG0PSPoUGAotrSyiFKmS1MW+IP/9lpL81cQwDInc2Z0dztJpmr6XHrUyeA0SOi2VgdJaVykjA8KXm39Ektw1ygN9Q4NQSmONKqWGgC3FU1BtHWG91X1Q1kBrq16jgA+224EKECz4IF2gGIOPILvOWVk2IvkcYlJlApoIlFrvwCtKipSMkH94gtRSuQxQ+l0ebL5GGaDWRBCoUuSjUVbo1la6ipi++5UGtL0PgNvOeoSH22833+8+f/v6AEwaqky8TIkcKN3Dule6YjxuUdEyWLfjIknTNEmIk3WxEVc2Zy/CGJAejEmS2tkWGukMei9iT46SIEzRf6vwwdLiNmRwF5G3E80kSUotvY9KM2PEl4E1v06APhXWUBTKqFAUjE6pzqBRVYVmOb9aZKDlDp1fLqbo+PF9h45xcUDxw1alWg9LuJ+v4A3cj3lWcDFliWuXq0PwWtvyxxB+XIvHrEg0cNJskGk0LObkkMP8hMIJ/A3hqacbcpd0Q/C9WmUwPlDFyxXPSDzx6ePNd8ZfVH8Fns/3+JzAR0QURxgMxJgQt/izj56Sml2MuXhyEJTaeCSzTHpuT5g7DL0zwMajJWs0bJ+XbTmPhDn8CXOA3yD/C+5nGXVwOMD9HMWDPDviqYKRLRK9dD3FFa3u0omVL53qBtdSwDiEUZNoCmB01tsFCdUbRdxb8JJGRJlNBu8q2abJ61Y5NUVsgsYBNaWP9F41kQh2bLfCfxUNxYliKkxqlWF7feJlHmX4aunuONSyXTyDUUF6Vq2IJNmRguikIyVoPjyjprRbzjF/y18hK4IjIU4cvJahbCj74qq4ers4IgJ2tDo7LDw2igbf2BD5Cpp95ZAm+9yiwyXnM6CbaYMhWj0GD9oiGyrxs/iOclDQkR4bM3Dh0DeyQ5ZfniO09f6gRYvSMDYkyfc1OVxc0OicgUgz8R86W2ycjB4NRbAF3bi4vHM9viwg1rL8Mfj5ZZ4ozC/Lg1Y0V5dnq6oeN36H+WwGS9LyXKtBL1JH2w2r0yH0Kf4+jz0+DUQU/R+QyOKqfga6HOoATxFDFsAivsa9Wf3seUrWSvY3MQl0Ojg8+R9QSwMEFAAAAAgA21UoXRcOReFKAwAAkgYAABYAHABzb2x1dGlvbnMvY2FuZGlkYXRlLnB5VVQJAAPeH6BqcSCganV4CwABBPUBAAAEAAAAAG1VXW/bOBB8169Y6HA4MpV4jov4IYAPaIteUSBtD0j6FBgKLa0sohSpktTFviD//ZaS/NXEMAyJ3NmdHc7SaZq+lx61MngNEjotlYHSWlcpIwPCl5t/RJLcNcoDfUODUEpjjSqlhoAtxVNQbR1hvdV9UNZAa6teo4APttuBChAs+CBdoBiDjyC7zllZNiL5HGJSZQKaCJRa78ArSoqUjJB/eILUUrkMUPpdHmy+Rhmg1kQQqFLko1FW6NZWuoqYvvuVBrS9D4DbznqEh9tvN9/vPn/7+gBMGqpMvEyJHCjdw7pXumI8blHRMli34yJJ0zRJiJN1sRFXNmcvwhiQHoxJktrZFhrpDHovYk+OkiBM0X+r8MHS4jZkcBeRtxPNJElKLb2PSjNjxJeBNb9OgD4V1lAUyqhQFIxOqc6gUVWFZjm/WmSg5Q6dXy6m6PjxfYeOcXFA8cNWpVoPS7ifr+AN3I95VnAxZYlrl6tD8Frb8scQflyLx6xINHDSbJBpNCzm5JDD/ITCCfwN4amnG3KXdEPwvVplMD5QxcsVz0g88enjzXfGX1R/BZ7P9/icwEdEFEcYDMSYELf4s4+ekppdjLl4chCU2ngks0x6bk+YOwy9M8DGoyVrNGyfl205j4Q5/AlzgN8g/wvuZxl1cDjA/RzFgzw74qmCkS0SvXQ9xRWt7tKJlS+d6gbXUsA4hFGTaApgdNbbBQnVG0XcW/CSRkSZTQbvKtmmyetWOTVFbILGATWlj/ReNZEIdmy3wn8VDcWJYipMapVhe33iZR5l+Grp7jjUsl08g1FBelatiCTZkYLopCMlaD48o6a0W84xf8tfISuCIyFOHLyWoWwo++KquHq7OCICdrQ6Oyw8NooG39gQ+QqafeWQJvvcosMl5zOgm2mDIVo9Bg/aIhsq8bP4jnJQ0JEeGzNw4dA3skOWX54jtPX+oEWL0jA2JMn3NTlcXNDonIFIM/EfOltsnIweDUWwBd24uLxzPb4sINay/DH4+WWeKMwvy4NWNFeXZ6uqHjd+h/lsBkvS8lyrQS9SR9sNq9Mh9Cn+Po89Pg1EFP0fkMjiqn4GuhzqAE8RQxbAIr7GvVn97HlK1kr2NzEJdDo4PPkfUEsDBAoAAAAAACNWKF0AAAAAAAAAAAAAAAAIABwALmNsYXVkZS9VVAkAA2EgoGrVIKBqdXgLAAEE9QEAAAQAAAAAUEsDBBQAAAAIACNWKF2xmrKMHQEAAKACAAAVABwALmNsYXVkZS9zZXR0aW5ncy5qc29uVVQJAANhIKBqcSCganV4CwABBPUBAAAEAAAAAHWST0/DMAzF7/0UUU8MjXLnNiRAHLiwww4IIbdx14g0qWKHUU377rjdn3asuzR6v1fbT4m3iVJpg6E2RMY7Sh/UVpBAjSVEy29eo8BUe8cL+k7nexes9RvhH70U8I6gD6aoF+vzkQrYDOpJGx7UKhjGQT4CVTdNy5V36q5WFQSHRBn+gI3AqMjbyF3Q+wKcNlpY1rTqdvavxdqwIgaONO1pU5bTDmh9bUpAQghF9eU8I2W1nqovfF3LIeEv28s9XFIGYy9p0YxS5EjcBZjONUv7ys/58eFcO36ZU7ImUjUxKIaJ8Zs18hldYf6MXFRnZNlfx2G8fHedmZJky/3vsEroILeoBXCIeNygyH7RbVE377Vc7otOf3Xdkl3yB1BLAwQUAAAACAAjVihdBcQCFjEAAAA3AAAACgAcAC5naXRpZ25vcmVVVAkAA2EgoGpxIKBqdXgLAAEE9QEAAAQAAAAAi48vqExOTM5IjY/X5yoqzSsGk3o5+elcSanFJXpZxfl5XMX5OaUlmflASbBYQSUXAFBLAwQUAAAACAAjVihdECd4a1kAAABrAAAAEQAcAHJlc2VhcmNoX25vdGVzLm1kVVQJAANhIKBqcSCganV4CwABBPUBAAAEAAAAABWLMQrDQAwEe71iIXU+kGe4cX2YJTqwJSMpxP59lGaYWdgHFiZHbArzYoqsMYtwI1I9CrSKGycDvJrz6OEl8oTep5cyZ3ZsOuzNlmB+9mr56iiU4/82XiU/UEsDBBQAAAAIACJWKF0HQlH2VAAAAFoAAAAJABwAQUdFTlRTLm1kVVQJAANgIKBqcSCganV4CwABBPUBAAAEAAAAAA2LPQ6AIAxGd0/xnYA7OBjj4mA8AARqJCmUlCbq7WV9PweFBD+vy366kjxCTbiEWR5kA70hGn8OmyF32E0DNdJcqBqaikkUHoMONwKlJj2b6OemH1BLAwQUAAAACAAjVihdB0JR9lQAAABaAAAACQAcAENMQVVERS5tZFVUCQADYSCganEgoGp1eAsAAQT1AQAABAAAAAANiz0OgCAMRndP8Z2AOzgY4+JgPAAEaiQplJQm6u1lfT8HhQQ/r8t+upI8Qk24hFkeZAO9IRp/Dpshd9hNAzXSXKgamopJFB6DDjcCpSY9m+jnph9QSwECHgMUAAAACAAiVihdwwCVlS0DAACWBgAACAAYAAAAAAABAAAApIEAAAAAQUdFTlQubWRVVAUAA2AgoGp1eAsAAQT1AQAABAAAAABQSwECHgMKAAAAAAAjVihdAAAAAAAAAAAAAAAACgAYAAAAAAAAAAAApIFvAwAAcnVucy5qc29ubFVUBQADYSCganV4CwABBPUBAAAEAAAAAFBLAQIeAwoAAAAAANtVKF0AAAAAAAAAAAAAAAAKABgAAAAAAAAAEADtQbMDAABzb2x1dGlvbnMvVVQFAAPeH6BqdXgLAAEE9QEAAAQAAAAAUEsBAh4DFAAAAAgA21UoXRcOReFKAwAAkgYAABkAGAAAAAAAAQAAAKSB9wMAAHNvbHV0aW9ucy9iYXNlbGluZV9tbHAucHlVVAUAA94foGp1eAsAAQT1AQAABAAAAABQSwECHgMUAAAACADbVShdFw5F4UoDAACSBgAAFgAYAAAAAAABAAAApIGUBwAAc29sdXRpb25zL2NhbmRpZGF0ZS5weVVUBQAD3h+ganV4CwABBPUBAAAEAAAAAFBLAQIeAwoAAAAAACNWKF0AAAAAAAAAAAAAAAAIABgAAAAAAAAAEADtQS4LAAAuY2xhdWRlL1VUBQADYSCganV4CwABBPUBAAAEAAAAAFBLAQIeAxQAAAAIACNWKF2xmrKMHQEAAKACAAAVABgAAAAAAAEAAACkgXALAAAuY2xhdWRlL3NldHRpbmdzLmpzb25VVAUAA2EgoGp1eAsAAQT1AQAABAAAAABQSwECHgMUAAAACAAjVihdBcQCFjEAAAA3AAAACgAYAAAAAAABAAAApIHcDAAALmdpdGlnbm9yZVVUBQADYSCganV4CwABBPUBAAAEAAAAAFBLAQIeAxQAAAAIACNWKF0QJ3hrWQAAAGsAAAARABgAAAAAAAEAAACkgVENAAByZXNlYXJjaF9ub3Rlcy5tZFVUBQADYSCganV4CwABBPUBAAAEAAAAAFBLAQIeAxQAAAAIACJWKF0HQlH2VAAAAFoAAAAJABgAAAAAAAEAAACkgfUNAABBR0VOVFMubWRVVAUAA2AgoGp1eAsAAQT1AQAABAAAAABQSwECHgMUAAAACAAjVihdB0JR9lQAAABaAAAACQAYAAAAAAABAAAApIGMDgAAQ0xBVURFLm1kVVQFAANhIKBqdXgLAAEE9QEAAAQAAAAAUEsFBgAAAAALAAsAlwMAACMPAAAAAA=='))) as zf:
        zf.extractall(PROJECT)
    os.symlink(FIXED / 'harness', PROJECT / 'harness', target_is_directory=True)
    for path in (FIXED / 'harness').glob('*.py'):
        path.chmod(0o444)
    subprocess.run(['git', 'init', '-b', 'main'], cwd=PROJECT, check=True)
    subprocess.run(['git', 'config', 'user.name', 'Fractal Researcher'], cwd=PROJECT, check=True)
    subprocess.run(['git', 'config', 'user.email', 'research@local'], cwd=PROJECT, check=True)
    subprocess.run(['git', 'add', '.'], cwd=PROJECT, check=True)
    subprocess.run(['git', 'commit', '-m', 'baseline'], cwd=PROJECT, check=True)
    print('Created:', PROJECT)


In [ ]:
%cd /content/fractalsearch
!find . -maxdepth 2 -type f | sort

## Look at the editable file

This plain MLP is the baseline. The coding agent can change the architecture, features, optimizer, learning rate, batch size, scheduler, sampling, and loss—as long as it remains a general learner.

In [ ]:
from pathlib import Path
print(Path('solutions/candidate.py').read_text())

## Run the baseline

Every scored experiment gets 60 seconds of training. Lower MSE is better. The evaluator also saves the exact source and three preview images.

In [ ]:
!python -m harness.evaluate solutions/candidate.py

## Results

Rerun this cell whenever you want a fresh progress view.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from IPython.display import display

rows = [json.loads(line) for line in Path('runs.jsonl').read_text().splitlines() if line.strip()]
good = [row for row in rows if row.get('status') == 'ok']

if not good:
    print('No completed runs yet.')
else:
    frame = pd.DataFrame(good)
    frame['run'] = range(1, len(frame) + 1)
    frame['best_so_far'] = frame['mse'].cummin()
    display(frame.nsmallest(10, 'mse')[['run', 'name', 'mse', 'mae', 'train_seconds', 'description']])

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(frame['run'], frame['mse'], 'o-', alpha=.45, label='run')
    axes[0].plot(frame['run'], frame['best_so_far'], linewidth=2, label='best so far')
    axes[0].set(xlabel='experiment', ylabel='validation MSE', title='Research progress')
    axes[0].legend()

    latest = Path('runs') / good[-1]['run_id']
    preview = Image.open(latest / 'prediction.png')
    axes[1].imshow(preview)
    axes[1].set_title('Latest prediction')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

    print('Latest run:')
    display(Image.open(latest / 'groundtruth.png'))
    display(Image.open(latest / 'prediction.png'))
    display(Image.open(latest / 'error.png'))

## Hand it to a coding agent

Open a terminal attached to this Colab runtime. Keep secrets in the terminal—never paste API keys into a notebook. The VM and its files disappear when Colab resets, and agent use may consume subscription limits or paid API credits.

If your Colab account does not show a terminal, run this part from a normal terminal environment instead; an interactive CLI is awkward inside a notebook cell.

### Codex CLI

<pre><code>curl -fsSL https://chatgpt.com/codex/install.sh | sh
export PATH="/root/.local/bin:$PATH"
codex login --device-auth
cd /content/fractalsearch
codex --sandbox workspace-write --ask-for-approval never \
  "Read AGENTS.md and run the experiment loop. Stop after five new scored experiments."
</code></pre>

Device login is made for remote/headless machines. If it is disabled for your account, follow the authentication options printed by `codex login`.

### Claude Code

<pre><code>curl -fsSL https://claude.ai/install.sh | bash
export PATH="/root/.local/bin:$PATH"
cd /content/fractalsearch
claude --permission-mode dontAsk \
  "Read CLAUDE.md and run the experiment loop. Stop after five new scored experiments."
</code></pre>

Claude Code will walk you through sign-in the first time. Its free web plan does not include Claude Code.

## While it runs

Let the terminal do its thing. Come back here and rerun the **Results** cell to see the curve and latest prediction.

The useful research record is not the final answer from the agent. It is the chain of hypotheses, diffs, failures, scores, and the best saved candidate.

## Keep your work

Colab storage is temporary. This creates one archive; it does not download anything automatically.

In [ ]:
!zip -qr /content/fractalsearch_results.zip solutions research_notes.md runs.jsonl best.json runs
print('Saved to /content/fractalsearch_results.zip')

---

Adapted for Colab from [MaxRobinsonTheGreat/fractalsearch](https://github.com/MaxRobinsonTheGreat/fractalsearch) and the experiment structure in [karpathy/autoresearch](https://github.com/karpathy/autoresearch). Codex installation/auth instructions follow [OpenAI's CLI documentation](https://learn.chatgpt.com/docs/codex/cli) and [headless login guidance](https://learn.chatgpt.com/docs/auth). Claude Code setup follows [Anthropic's documentation](https://code.claude.com/docs/en/setup).